**Name:** Teesha Ramchandani 
**ID:** 12342260

# Lab Assignment 7
## RNN Applications: from SimpleRNN to LSTM, and from Classification to Tagging


---

### Objective

The tutorial ended with a problem rather than a solution. A **SimpleRNN** classified sentiment at
about 64%, and its accuracy **collapsed on long sentences** — 70.5% on short ones, 58.4% on long
ones. The lecture blames the **vanishing gradient**, and claims an **LSTM** fixes it.

This assignment does two things.

**Part A — Task 1.** Test that claim properly. Same data, same task, same settings; you change
**only** the recurrent layer. If the lecture is right, the LSTM should help — and it should help
**most on the long sentences**.

**Part B — Tasks 2 to 5.** Take the LSTM to a different kind of problem: **POS tagging**, where
every word gets its own label instead of one label for the whole sentence. Then make it
**bidirectional** and measure what that buys you.

You tagged parts of speech once before, in **Lab 3, using a Hidden Markov Model**. This time the
model learns the patterns from data instead of from explicit transition and emission counts.

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Application | sentiment classification | sentiment (Task 1), then **POS tagging** (Tasks 2–5) |
| Shape | many-to-one | many-to-one, then **many-to-many** |
| Recurrent cell | `SimpleRNN` | **`LSTM`**, then **`Bidirectional(LSTM)`** |
| Output | `Dense(1)` on the final state | `Dense(1)`, then `TimeDistributed(Dense(n_tags))` |
| Data | movie review sentences | the same reviews, then the **Penn Treebank** |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Does the LSTM actually fix it? | **4** |
| **Task 2** | Data preparation for tagging | **4** |
| **Task 3** | Building and training the LSTM tagger | **4** |
| **Task 4** | Evaluating the tagger correctly | **4** |
| **Task 5** | Bidirectional LSTM and comparison | **4** |
| | **Total** | **20** |

---

### Before you start

- A GPU is **not** required. Everything here runs in a few minutes on CPU.
- **Task 1 needs the file `mini_sentiment.tsv`**, the same one the tutorial used. You must have it
  in the same folder as this notebook before you start.
  **In Colab:** click the folder icon in the left sidebar and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Do **not** rebuild this file from NLTK yourself. The baseline numbers you are comparing against
  come from this exact file in this exact order, so a regenerated copy would make the comparison
  invalid.
- You may reuse the tutorial's Task 1 pipeline directly — the marks are for the **measurement and
  the explanation**, not for retyping the data loading.
- **Tasks 2–5** use the Penn Treebank from NLTK.
- Use the **universal tagset** (12 tags), not the full Penn tagset (45 tags). With only 3,914
  sentences the rarer Penn tags have far too few examples.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

These are the only values given to you.

In [1]:
import numpy as np
import nltk
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Bidirectional, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)
from nltk.corpus import treebank

keras.utils.set_random_seed(42)

# --- Task 1 (sentiment), same values as the tutorial ---
MAX_WORDS = 5000
SENT_LEN  = 40

# --- Tasks 2-5 (POS tagging) ---
MAX_LEN   = 50      # pad/truncate every sentence to this many words
EMBED_DIM = 64
RNN_UNITS = 64
N_TRAIN   = 3000    # first 3000 sentences for training, the rest for testing

tagged_sentences = treebank.tagged_sents(tagset='universal')
print(f"{len(tagged_sentences)} tagged sentences")
print("example:", tagged_sentences[0][:6])

3914 tagged sentences
example: [('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ')]


---
# Task 1: Does the LSTM actually fix it? &nbsp;&nbsp;*(4 marks)*

The tutorial's SimpleRNN scored this on the sentiment test set:

| Group | n | SimpleRNN |
|---|---|---|
| SHORT (≤ 15 words) | 808 | **0.705** |
| ALL | 2662 | **0.639** |
| LONG (≥ 28 words) | 680 | **0.584** |

Rebuild that classifier **changing only the recurrent layer** — `SimpleRNN(32)` becomes
`LSTM(32)`. Everything else stays identical: same file, same `MAX_WORDS`, same `SENT_LEN`, same
8000/2662 split, same epochs, same batch size. If you change anything else the comparison is
meaningless.

Then score it the same three ways and put the numbers side by side.

### What your code must create

| Name | Is |
|---|---|
| `X_sent`, `y_sent` | the padded sentiment data, exactly as in the tutorial |
| `raw_len` | each test sentence's true word count, **before** padding |
| `model_lstm_sent` | the same architecture with `LSTM(32)` in place of `SimpleRNN(32)` |

### Output your code must print

```
GROUP                     n    SimpleRNN      LSTM     change
------------------------------------------------------------
SHORT (<= 15 words)     808        0.705     0.___     +_.___
ALL                    2662        0.639     0.___     +_.___
LONG  (>= 28 words)     680        0.584     0.___     +_.___
```

### Then answer

**Q1.** Where is the LSTM's improvement largest — on short sentences or long ones? Does that match
what the lecture predicts about vanishing gradients? Two or three sentences.

### Hints

- Read `mini_sentiment.tsv` exactly as the tutorial does: one header line, then `pos`/`neg` and the
  sentence separated by a tab.
- The split is the first 8000 sentences for training, the remainder for testing.
- `raw_len` must come from the token lists, **not** from the padded array — after padding every
  sentence has the same length.
- Call `keras.utils.set_random_seed(42)` before building the model.

In [2]:
# ---------------- TASK 1 ----------------

# TODO 1: load mini_sentiment.tsv, build the vocabulary, encode and pad -> X_sent, y_sent
#         (same procedure as the tutorial)
labels_raw, texts = [], []
with open('mini_sentiment.tsv', encoding='utf-8') as f:
    next(f)                                     # skip the header row
    for line in f:
        label, sentence = line.rstrip('\n').split('\t')
        labels_raw.append(1 if label == 'pos' else 0)
        texts.append(sentence.split())

y_sent = np.array(labels_raw)

counts = {}
for t in texts:
    for w in t:
        counts[w] = counts.get(w, 0) + 1

vocab = ['<pad>', '<unk>'] + [w for w, _ in sorted(counts.items(), key=lambda kv: -kv[1])[:MAX_WORDS]]
word_index_sent = {w: i for i, w in enumerate(vocab)}

sequences = [[word_index_sent.get(w, 1) for w in t] for t in texts]      # 1 = <unk>
X_sent = pad_sequences(sequences, maxlen=SENT_LEN)                      # 0 = <pad>

# TODO 2: compute raw_len for the TEST sentences, from the token lists before padding
raw_len = np.array([len(t) for t in texts])[8000:]

# TODO 3: build the same model as the tutorial but with LSTM(32) instead of SimpleRNN(32),
#         and train it with identical settings
X_train, y_train = X_sent[:8000], y_sent[:8000]
X_test,  y_test  = X_sent[8000:], y_sent[8000:]

keras.utils.set_random_seed(42)

model_lstm_sent = Sequential([
    Input(shape=(SENT_LEN,)),
    Embedding(len(vocab), 32),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
model_lstm_sent.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_sent = model_lstm_sent.fit(X_train, y_train, epochs=4, batch_size=64,
                                    validation_split=0.2, verbose=0)

# TODO 4: score it three ways -- SHORT (<=15 words), ALL, LONG (>=28 words)
pred_sent = (model_lstm_sent.predict(X_test, verbose=0).ravel() > 0.5).astype(int)

short = raw_len <= 15
long_ = raw_len >= 28

acc_short_lstm = (pred_sent[short] == y_test[short]).mean()
acc_all_lstm   = (pred_sent == y_test).mean()
acc_long_lstm  = (pred_sent[long_] == y_test[long_]).mean()

# TODO 5: print the comparison table against the SimpleRNN numbers given above
simplernn_short, simplernn_all, simplernn_long = 0.705, 0.639, 0.584

print(f"{'GROUP':<20}{'n':>6}{'SimpleRNN':>12}{'LSTM':>10}{'change':>11}")
print("-" * 62)
print(f"{'SHORT (<= 15 words)':<20}{short.sum():>6}{simplernn_short:>12.3f}{acc_short_lstm:>10.3f}{acc_short_lstm-simplernn_short:>+11.3f}")
print(f"{'ALL':<20}{len(y_test):>6}{simplernn_all:>12.3f}{acc_all_lstm:>10.3f}{acc_all_lstm-simplernn_all:>+11.3f}")
print(f"{'LONG  (>= 28 words)':<20}{long_.sum():>6}{simplernn_long:>12.3f}{acc_long_lstm:>10.3f}{acc_long_lstm-simplernn_long:>+11.3f}")


GROUP                    n   SimpleRNN      LSTM     change
--------------------------------------------------------------
SHORT (<= 15 words)    808       0.705     0.776     +0.071
ALL                   2662       0.639     0.740     +0.101
LONG  (>= 28 words)    680       0.584     0.694     +0.110


---
# Task 2: Data preparation for tagging &nbsp;&nbsp;*(4 marks)*

Now switch tasks. Turn the tagged sentences into the rectangular integer arrays an RNN needs.

Each sentence is a list of `(word, tag)` pairs, so you need **two** parallel sequences: word ids
and tag ids, aligned position by position.

### What your code must create

| Name | Holds |
|---|---|
| `words`, `tags` | the sorted vocabularies (lowercase the words) |
| `word_index`, `tag_index` | word → id and tag → id dictionaries |
| `X`, `y` | padded word-id and tag-id arrays, shape `(n_sentences, MAX_LEN)` |
| `X_train`, `y_train`, `X_test`, `y_test` | split at `N_TRAIN` |

### Output your code must print

```
Vocabulary size: ...
Number of tags: ...
Tags: ['.', 'ADJ', 'ADP', ...]
X shape: (3914, 50)   y shape: (3914, 50)
Training sentences: 3000   Test sentences: 914
```
Then print the first 8 words and tags of one sentence, in both word form and id form, so it is
visible that the two sequences line up.

### Hints

- Reserve **id 0 for padding** in both vocabularies, and **id 1 for unknown words**. Real word ids
  start at 2, real tag ids at 1. Nothing may be given id 0 except padding.
- Lowercase words when building the vocabulary, and lowercase again when looking them up.
- Use `padding='post'` here. The tutorial padded at the *front*, which is fine when only the last
  hidden state matters — but for tagging, padding at the **end** keeps word `i` and tag `i` at the
  same index, which is what makes masking work.
- Pad `X` and `y` **the same way**.
- Check what shape `sparse_categorical_crossentropy` expects for the targets.

In [3]:
# ---------------- TASK 2 ----------------

# TODO 6: build the word and tag vocabularies (`words`, `tags`), lowercasing the words
word_counts = set()
tag_counts = set()
for sent in tagged_sentences:
    for w, t in sent:
        word_counts.add(w.lower())
        tag_counts.add(t)

words = sorted(word_counts)
tags = sorted(tag_counts)

# TODO 7: build `word_index` and `tag_index`, leaving id 0 free for padding
#         (and id 1 for unknown words in word_index)
word_index = {w: i + 2 for i, w in enumerate(words)}   # 0 = pad, 1 = unk
tag_index  = {t: i + 1 for i, t in enumerate(tags)}    # 0 = pad
index_tag  = {i: t for t, i in tag_index.items()}

# TODO 8: convert every sentence into a list of word ids and a list of tag ids
word_seqs = [[word_index[w.lower()] for w, t in sent] for sent in tagged_sentences]
tag_seqs  = [[tag_index[t] for w, t in sent] for sent in tagged_sentences]

# TODO 9: pad both to MAX_LEN with padding='post', giving `X` and `y`
X = pad_sequences(word_seqs, maxlen=MAX_LEN, padding='post', value=0)
y = pad_sequences(tag_seqs, maxlen=MAX_LEN, padding='post', value=0)

# TODO 10: split into X_train, y_train, X_test, y_test at N_TRAIN
X_train, y_train = X[:N_TRAIN], y[:N_TRAIN]
X_test,  y_test  = X[N_TRAIN:], y[N_TRAIN:]

# TODO 11: print the statistics above, plus one example sentence in word form
#          and id form so the alignment is visible
print(f"Vocabulary size: {len(word_index) + 2}")
print(f"Number of tags: {len(tags)}")
print(f"Tags: {tags}")
print(f"X shape: {X.shape}   y shape: {y.shape}")
print(f"Training sentences: {len(X_train)}   Test sentences: {len(X_test)}")

example_words = [w for w, t in tagged_sentences[0]][:8]
example_tags  = [t for w, t in tagged_sentences[0]][:8]
print("\nwords (form):", example_words)
print("tags  (form):", example_tags)
print("words (ids) :", [word_index[w.lower()] for w in example_words])
print("tags  (ids) :", [tag_index[t] for t in example_tags])


Vocabulary size: 11389
Number of tags: 12
Tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
X shape: (3914, 50)   y shape: (3914, 50)
Training sentences: 3000   Test sentences: 914

words (form): ['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will']
tags  (form): ['NOUN', 'NOUN', '.', 'NUM', 'NOUN', 'ADJ', '.', 'VERB']
words (ids) : [7986, 10981, 460, 1263, 11343, 7554, 460, 11217]
tags  (ids) : [7, 7, 1, 8, 7, 2, 1, 11]


---
# Task 3: Building and training the LSTM tagger &nbsp;&nbsp;*(4 marks)*

```
   word ids  ->  Embedding  ->  LSTM(return_sequences=True)  ->  TimeDistributed(Dense(softmax))
   (n, 50)       (n, 50, 64)         (n, 50, 64)                        (n, 50, n_tags+1)
```

Compare this with the tutorial's model. Two things changed, and both are essential:

1. **`return_sequences=True`** — the LSTM must output a hidden state at **every** time step, not
   just the last. Without it you get one vector for the whole sentence and cannot tag words.
2. **`TimeDistributed(Dense(...))`** — applies the same Dense classifier independently at every
   time step. One shared classifier, applied 50 times.

### What your code must create

| Name | Is |
|---|---|
| `model_lstm` | the compiled tagger |
| `history_lstm` | the return value of `model_lstm.fit(...)` |

### Output your code must print

- `model_lstm.summary()` — showing **real output shapes and parameter counts**, not `?` and `0`
- the per-epoch table, ending with the final `loss`, `accuracy`, `val_loss`, `val_accuracy`

A correct summary looks like this (total parameters **762,765**):

```
 embedding (Embedding)        (None, 50, 64)        728,896
 lstm (LSTM)                  (None, 50, 64)         33,024
 time_distributed             (None, 50, 13)            845
```

Read the middle column: the LSTM keeps all 50 positions, and the output has 13 values per position
— one score per tag. That is what `return_sequences=True` and `TimeDistributed` buy you.

### Hints

- **Start your `Sequential` with `Input(shape=(MAX_LEN,))`.** Without it Keras does not build the
  model until training begins, so `model.summary()` prints `?` for every output shape and
  `0 (unbuilt)` for every parameter count — useless for your submission. The same applies to your
  Task 1 model, with `SENT_LEN` instead.
- Use `Embedding(..., mask_zero=True)` so later layers ignore padded positions and the model is not
  trained to predict a tag for padding. Note what this does to the accuracy Keras reports — Task 4
  asks about it.
- The output layer needs one unit per tag id **including the reserved 0**, so its size is
  `len(tags) + 1`.
- Loss `sparse_categorical_crossentropy` (targets are integer ids, not one-hot), optimizer `adam`.
- About 5 epochs, `batch_size=32`, `validation_split=0.2`. More will overfit; the dataset is small.

In [4]:
# ---------------- TASK 3 ----------------

# TODO 12: build `model_lstm` -- Embedding(mask_zero=True), LSTM(return_sequences=True),
#          TimeDistributed(Dense(..., activation='softmax'))
keras.utils.set_random_seed(42)

n_tags = len(tags)

model_lstm = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(len(word_index) + 2, EMBED_DIM, mask_zero=True),
    LSTM(RNN_UNITS, return_sequences=True),
    TimeDistributed(Dense(n_tags + 1, activation='softmax'))
])

# TODO 13: compile it with the right loss and optimizer
model_lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# TODO 14: print model_lstm.summary()
model_lstm.summary()

# TODO 15: train it, keeping the result in `history_lstm`
history_lstm = model_lstm.fit(X_train, y_train, epochs=5, batch_size=32,
                               validation_split=0.2, verbose=0)

# TODO 16: print the per-epoch table and the final epoch's four metrics
h = history_lstm.history
print(f"{'epoch':>6}{'loss':>9}{'accuracy':>10}{'val_loss':>10}{'val_accuracy':>14}")
for e in range(len(h['loss'])):
    print(f"{e+1:>6}{h['loss'][e]:>9.4f}{h['accuracy'][e]:>10.4f}"
          f"{h['val_loss'][e]:>10.4f}{h['val_accuracy'][e]:>14.4f}")

print(f"\nFinal epoch -- loss: {h['loss'][-1]:.4f}  accuracy: {h['accuracy'][-1]:.4f}  "
      f"val_loss: {h['val_loss'][-1]:.4f}  val_accuracy: {h['val_accuracy'][-1]:.4f}")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 64)         │       728,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 50, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 50, 13)         │           845 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 762,765 (2.91 MB)

 Trainable params: 762,765 (2.91 MB)

 Non-trainable params: 0 (0.00 B)

 epoch     loss  accuracy  val_loss  val_accuracy
     1   2.1123    0.3695    1.6084        0.5527
     2   1.0533    0.7153    0.6996        0.8121
     3   0.4507    0.8813    0.3893        0.9024
     4   0.2200    0.9509    0.2914        0.9184
     5   0.1341    0.9705    0.2551        0.9229

Final epoch -- loss: 0.1341  accuracy: 0.9705  val_loss: 0.2551  val_accuracy: 0.9229


---
# Task 4: Evaluating the tagger correctly &nbsp;&nbsp;*(4 marks)*

> **Watch out — this is the trap in this assignment.** Most positions in `y_test` are **padding**,
> not real words. An accuracy computed over the whole array is meaningless: it is dominated by
> padding, and depending on how you compute it you can get a number that is far too high *or* far
> too low. Neither tells you how good your tagger is.
>
> **Report accuracy over real tokens only** — every position where the true tag id is not the
> padding id. Compute it yourself; do not simply quote whatever `model.evaluate()` prints.

### Part A — masked accuracy

Write `masked_accuracy(model, X, y)` returning the fraction of **non-padding** positions where the
predicted tag id equals the true one. Also print the accuracy `model_lstm.evaluate(...)` gives, and
**state in one line whether the two agree and why**.

### Part B — per-tag accuracy

Report accuracy separately for each of the 12 tags, sorted worst first. This shows *which* parts of
speech are hard, which is far more useful than a single number.

### Part C — tag new sentences

Write `tag_sentence(model, sentence)` taking a plain string and returning `(word, predicted_tag)`
pairs. Run it on:

* `The quick brown fox jumps over the lazy dog .`
* `He will book a flight to New York tomorrow .`
* `They are reading a book about the stock market .`

### Output your code must print

```
Masked accuracy (real tokens only): 0.____
Keras evaluate() accuracy:          0.____
[one line explaining whether they agree and why]

PER-TAG ACCURACY
tag      n      accuracy
...

The/DET quick/ADJ brown/ADJ fox/NOUN ...
```

### Hints

- `model.predict(X)` returns `(n, MAX_LEN, n_tags+1)`. Use `.argmax(-1)` for predicted ids.
- Build the mask as `y_true != 0` and compare only those positions.
- Words missing from `word_index` must map to the unknown id, exactly as in Task 2.
- Sentences 2 and 3 are chosen deliberately: **"book"** is a verb in one and a noun in the other.
  Check whether your tagger gets both right.
- Sentence 1 is ordinary English, but the Treebank is **Wall Street Journal text**. Expect some odd
  tags, and think about why.

In [5]:
# ---------------- TASK 4 ----------------

def masked_accuracy(model, X, y):
    """
    Tagging accuracy over REAL tokens only, ignoring padded positions.

    Output: float
    """
    # TODO 17: predict, then take argmax over the tag axis
    preds = model.predict(X, verbose=0).argmax(-1)
    # TODO 18: build a mask selecting positions where the TRUE tag is not padding
    mask = y != 0
    # TODO 19: return the accuracy over masked positions only
    return (preds[mask] == y[mask]).mean()


def tag_sentence(model, sentence):
    """
    Tag a plain string.

    Input :  sentence (str)
    Output:  list of (word, predicted_tag) pairs
    """
    # TODO 20: split and lowercase the sentence, map words to ids (unknown -> the <unk> id)
    orig_words = sentence.split()
    ids = [word_index.get(w.lower(), 1) for w in orig_words]

    # TODO 21: pad to MAX_LEN the same way as Task 2, predict, take argmax
    padded = pad_sequences([ids], maxlen=MAX_LEN, padding='post', value=0)
    pred_ids = model.predict(padded, verbose=0).argmax(-1)[0]

    # TODO 22: map predicted ids back to tag strings, pair them with the ORIGINAL words,
    #          and return only as many tags as there are words
    pred_tags = [index_tag.get(i, '?') for i in pred_ids[:len(orig_words)]]
    return list(zip(orig_words, pred_tags))


# TODO 23: print the masked accuracy and model_lstm.evaluate()'s accuracy, plus one line
#          saying whether they agree and why
masked_acc_lstm = masked_accuracy(model_lstm, X_test, y_test)
eval_loss, eval_acc = model_lstm.evaluate(X_test, y_test, verbose=0)

print(f"Masked accuracy (real tokens only): {masked_acc_lstm:.4f}")
print(f"Keras evaluate() accuracy:          {eval_acc:.4f}")
print("They agree closely because Embedding(mask_zero=True) makes Keras itself ignore "
      "padded positions when computing loss/accuracy, so evaluate() is already masked.")

# TODO 24: print per-tag accuracy for all 12 tags, worst first
preds_test = model_lstm.predict(X_test, verbose=0).argmax(-1)
mask_test = y_test != 0

per_tag = []
for t in tags:
    tid = tag_index[t]
    tag_mask = mask_test & (y_test == tid)
    n = tag_mask.sum()
    acc = (preds_test[tag_mask] == y_test[tag_mask]).mean() if n > 0 else float('nan')
    per_tag.append((t, n, acc))

per_tag.sort(key=lambda x: x[2])

print("\nPER-TAG ACCURACY")
print(f"{'tag':<8}{'n':>8}{'accuracy':>12}")
for t, n, acc in per_tag:
    print(f"{t:<8}{n:>8}{acc:>12.4f}")

# TODO 25: tag the three sentences above and print them as word/TAG
print()
test_sentences = [
    "The quick brown fox jumps over the lazy dog .",
    "He will book a flight to New York tomorrow .",
    "They are reading a book about the stock market .",
]
lstm_tagged = []
for s in test_sentences:
    tagged = tag_sentence(model_lstm, s)
    lstm_tagged.append(tagged)
    print(' '.join(f"{w}/{tg}" for w, tg in tagged))


Masked accuracy (real tokens only): 0.9305
Keras evaluate() accuracy:          0.9305
They agree closely because Embedding(mask_zero=True) makes Keras itself ignore padded positions when computing loss/accuracy, so evaluate() is already masked.

PER-TAG ACCURACY
tag            n    accuracy
ADJ         1410      0.7610
NUM         1208      0.8071
ADV          594      0.8114
NOUN        6797      0.9151
VERB        3034      0.9272
DET         1955      0.9719
ADP         2288      0.9812
PRT          777      0.9833
X           1498      0.9973
PRON         407      0.9975
CONJ         499      0.9980
.           2576      0.9984

The/DET quick/ADJ brown/NOUN fox/NOUN jumps/NOUN over/ADP the/DET lazy/NOUN dog/NOUN ./.
He/PRON will/VERB book/VERB a/DET flight/NOUN to/PRT New/ADJ York/NOUN tomorrow/NOUN ./.
They/PRON are/VERB reading/NOUN a/DET book/NOUN about/ADP the/DET stock/NOUN market/NOUN ./.


---
# Task 5: Bidirectional LSTM and comparison &nbsp;&nbsp;*(4 marks)*

### The idea

Your LSTM reads strictly **left to right**. When it tags a word it has seen everything before it
and **nothing after it**. But the disambiguating words are often on the right:

> *He will **book** a flight.* &nbsp;&nbsp;vs&nbsp;&nbsp; *She read a **book** about markets.*

A **bidirectional** RNN runs two LSTMs — one forward, one backward — and joins their hidden states
at each position:

$$h_t = \left[\overrightarrow{h_t}\; ;\; \overleftarrow{h_t}\right]$$

So every word is tagged using the whole sentence. Wrapping a layer in `Bidirectional(...)` is all
Keras needs; note that it **doubles** the output width per step.

> Bidirectionality is only possible because the **entire** sentence is available up front. You
> could not do this in a language model that must predict the next word — which is why Lab 6 did
> not use it.

### What your code must create

| Name | Is |
|---|---|
| `model_bilstm` | Task 3's model with the LSTM wrapped in `Bidirectional` |
| `history_bilstm` | its training history |

Train it **exactly** as you trained the LSTM — same epochs, batch size and split — or the
comparison is not fair.

### Output your code must print

```
MODEL COMPARISON (masked accuracy on the test set)
  LSTM      params ______   accuracy 0.____
  BiLSTM    params ______   accuracy 0.____
  improvement: +_.__ percentage points

[the three sentences tagged by the BiLSTM]
[any words where the two models disagree]
```

### Then answer

**Q2.** How many percentage points did bidirectionality gain, and how many extra parameters did it
cost? Was it worth it?

**Q3.** List every word where the two models disagree. Pick one and explain, using right-hand
context, why seeing the rest of the sentence could change the decision. *(Look closely at the
proper noun in sentence 2.)* Also say whether **"book"** needed bidirectionality at all, or whether
left context was already enough.

**Q4.** From your per-tag table, which tags were hardest? Sentence 1 is plain English yet gets
several tags wrong — what does that tell you about the data the model was trained on?

### Hints

- Only one line of the model changes. Training, evaluation and `masked_accuracy` are reused as is.
- Call `keras.utils.set_random_seed(42)` again before building the second model, so the comparison
  is not distorted by different random initialisation.
- `model.count_params()` gives the parameter count.
- To find disagreements, tag the same sentence with both models and compare position by position.

In [6]:
# ---------------- TASK 5 ----------------

# TODO 26: build `model_bilstm` -- identical to Task 3 but with the LSTM wrapped
#          in Bidirectional(...)
keras.utils.set_random_seed(42)

model_bilstm = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(len(word_index) + 2, EMBED_DIM, mask_zero=True),
    Bidirectional(LSTM(RNN_UNITS, return_sequences=True)),
    TimeDistributed(Dense(n_tags + 1, activation='softmax'))
])

# TODO 27: compile and train it with EXACTLY the same settings, keeping `history_bilstm`
model_bilstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_bilstm = model_bilstm.fit(X_train, y_train, epochs=5, batch_size=32,
                                   validation_split=0.2, verbose=0)

# TODO 28: print the comparison -- parameter count and masked accuracy for both models,
#          and the improvement in percentage points
masked_acc_bilstm = masked_accuracy(model_bilstm, X_test, y_test)

print("MODEL COMPARISON (masked accuracy on the test set)")
print(f"  LSTM      params {model_lstm.count_params():>8}   accuracy {masked_acc_lstm:.4f}")
print(f"  BiLSTM    params {model_bilstm.count_params():>8}   accuracy {masked_acc_bilstm:.4f}")
print(f"  improvement: {(masked_acc_bilstm - masked_acc_lstm) * 100:+.2f} percentage points")

# TODO 29: tag the three sentences from Task 4 with the BiLSTM and print them
print()
bilstm_tagged = []
for s in test_sentences:
    tagged = tag_sentence(model_bilstm, s)
    bilstm_tagged.append(tagged)
    print(' '.join(f"{w}/{tg}" for w, tg in tagged))

# TODO 30: print any positions where the two models predict different tags
print("\nDISAGREEMENTS (LSTM vs BiLSTM)")
for s, lstm_t, bilstm_t in zip(test_sentences, lstm_tagged, bilstm_tagged):
    print(f"\nSentence: {s}")
    disagreements = [(w1, t1, t2) for (w1, t1), (w2, t2) in zip(lstm_t, bilstm_t) if t1 != t2]
    if disagreements:
        for w, t1, t2 in disagreements:
            print(f"  {w:<12} LSTM={t1:<6} BiLSTM={t2}")
    else:
        print("  (no disagreements)")


MODEL COMPARISON (masked accuracy on the test set)
  LSTM      params   762765   accuracy 0.9305
  BiLSTM    params   796621   accuracy 0.9474
  improvement: +1.69 percentage points

The/DET quick/ADJ brown/NOUN fox/VERB jumps/NOUN over/ADP the/DET lazy/NOUN dog/NOUN ./.
He/PRON will/VERB book/VERB a/DET flight/NOUN to/PRT New/NOUN York/NOUN tomorrow/NOUN ./.
They/PRON are/VERB reading/VERB a/DET book/NOUN about/ADP the/DET stock/NOUN market/NOUN ./.

DISAGREEMENTS (LSTM vs BiLSTM)

Sentence: The quick brown fox jumps over the lazy dog .
  fox          LSTM=NOUN   BiLSTM=VERB

Sentence: He will book a flight to New York tomorrow .
  New          LSTM=ADJ    BiLSTM=NOUN

Sentence: They are reading a book about the stock market .
  reading      LSTM=NOUN   BiLSTM=VERB


### Your answers

**Q1.** Where the LSTM's improvement was largest, and whether that matches the vanishing-gradient
explanation:

*Your answer: The LSTM's improvement over the SimpleRNN was largest on the **LONG sentences**
(+0.110, from 0.584 to 0.694), larger than on ALL sentences (+0.101, 0.639 → 0.740) and smallest on
the **SHORT sentences** (+0.071, 0.705 → 0.776). This matches the vanishing-gradient theory
exactly: the SimpleRNN struggled most on long sentences because gradients shrink with every extra
time step they have to travel through during training, making it hard to learn from words far from
the end of a long sentence. The LSTM's gating mechanism gives the gradient a path that doesn't
shrink at every step, so it recovers the most accuracy precisely where the SimpleRNN was weakest —
on long sentences — confirming the lecture's claim.*

**Q2.** Gain from bidirectionality, and its cost in parameters:

*Your answer: Bidirectionality gained **+1.69 percentage points** of masked accuracy (93.05% for
the LSTM vs. 94.74% for the BiLSTM), at a cost of **33,856 extra parameters** (762,765 → 796,621,
about a 4.4% increase). That is a solid accuracy gain for a small parameter cost, so it was worth
it here — `Bidirectional` only doubles the recurrent layer's weights, not the much larger
embedding table, which is why the total parameter increase is small relative to the accuracy
gained.*

**Q3.** Where the two models disagree, and what right-hand context explains it:

*Your answer: The two models disagreed on three words: **"fox"** (LSTM: NOUN, BiLSTM: VERB) and
**"reading"** (LSTM: NOUN, BiLSTM: VERB), and **"New"** (LSTM: ADJ, BiLSTM: NOUN) in "He will book
a flight to New York tomorrow .". The clearest case is "New": reading strictly left-to-right, "New"
looks like an ordinary adjective (as in "a new flight"), so the unidirectional LSTM tags it ADJ.
Only by looking rightward to "York" can the tagger recognise "New York" as a single proper-noun
compound — which is exactly what the BiLSTM does, correctly tagging "New" as NOUN. As for "book":
both models tagged it VERB in "He will book a flight" and NOUN in "reading a book about the stock
market" — they **agree** on both instances, so for this word left context alone (the preceding
modal "will" vs. the determiner "a") was already enough to disambiguate it; bidirectionality was
not actually needed for "book" in these examples.*

**Q4.** The hardest tags, and what sentence 1 reveals about the training data:

*Your answer: The hardest tags by masked per-token accuracy were **ADJ (76.1%)**, **NUM (80.7%)**,
and **ADV (81.1%)** — all well below the other tags, which mostly scored above 90%. Sentence 1,
"The quick brown fox jumps over the lazy dog .", is plain, simple English, yet the BiLSTM still
mistags "fox" as VERB, and the words "quick"/"brown"/"lazy" belong to exactly the tag class (ADJ)
that scored worst overall. This reveals that the Penn Treebank sample used for training is Wall
Street Journal financial news text, not general-purpose English: it contains very few generic
descriptive adjectives, animal nouns, or simple declarative sentences of this kind, so the model
has seen too few examples of this vocabulary and sentence style to tag it reliably. The model's
weak performance on an "easy" pangram sentence is a symptom of a training-data domain mismatch, not
a flaw in the architecture.*

---
## Submission checklist

- [x] `Runtime → Restart and run all` completes without errors
- [x] **Task 1** — the three-way comparison table against the SimpleRNN baseline
- [x] **Task 2** — statistics printed, and one example shown in both word and id form
- [x] **Task 3** — `model_lstm.summary()` and the per-epoch table are visible
- [x] **Task 4** — masked accuracy, the comparison with `evaluate()`, the per-tag table, and the
      three tagged sentences
- [x] **Task 5** — both models compared on parameters and accuracy, BiLSTM tagging shown,
      disagreements listed
- [x] All four questions answered
- [x] Submit the `.ipynb` with all outputs saved

### Common mistakes

1. Changing something other than the recurrent layer in Task 1, making the comparison meaningless.
2. Using id 0 for a real word or a real tag — 0 must mean padding and nothing else.
3. Padding at the front (`padding='pre'`) for tagging, which misaligns words and tags.
4. Forgetting `return_sequences=True`, so the LSTM emits one vector instead of one per word.
5. Reporting raw accuracy over the whole padded array instead of over real tokens only.
6. Training the two taggers for different numbers of epochs.
7. Looking words up without lowercasing them, when the vocabulary was built lowercased.